# Component attribution across dynamical regimes

This notebook is a cleaned record of the original training program. Execution outputs and notebook execution metadata were removed; computational cells were preserved to maintain the historical experiment protocol. Dataset access and optional cache paths may require configuration.


In [ ]:

# ============================================================
# 0. Runtime / protocol
# ============================================================
import os, sys, math, time, json, hashlib, shutil, subprocess, zipfile, copy, gc
from pathlib import Path
from collections import OrderedDict
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DEVICE.type=="cuda", "A3-R1 requires CUDA."

ROOT=Path(os.getenv("CIDM_BENCH_ROOT",str(Path.cwd())))
OUT=ROOT/"CIDM_Foundation_A3_R1_ComponentAttribution_RESULTS"
FROZEN=OUT/"frozen"
CKPT=OUT/"checkpoints"
CACHE_ROOT=OUT/"restored_A2_caches"
BASE_REF=OUT/"A2_reference_results"
FIG=OUT/"figures"
FIGDATA=OUT/"figure_data"
PRED=OUT/"prediction_examples"

for p in [OUT,FROZEN,CKPT,CACHE_ROOT,BASE_REF,FIG,FIGDATA,PRED]:
    p.mkdir(parents=True,exist_ok=True)

HISTORY=4
HORIZON=6
SEEDS=[17,41,73]
SUPP_SEEDS=[17]
LR=1e-3
WEIGHT_DECAY=1e-4

MAX_UPDATES=int(os.getenv("CIDM_A3_R1_MAX_UPDATES","2500"))
MIN_UPDATES=int(os.getenv("CIDM_A3_R1_MIN_UPDATES","1000"))
EVAL_EVERY=int(os.getenv("CIDM_A3_R1_EVAL_EVERY","100"))
PATIENCE_EVALS=int(os.getenv("CIDM_A3_R1_PATIENCE_EVALS","6"))
REL_MIN_DELTA=float(os.getenv("CIDM_A3_R1_REL_MIN_DELTA","0.001"))
MICRO_STEPS=int(os.getenv("CIDM_A3_R1_MICRO_STEPS","30"))
BATCH_1D=int(os.getenv("CIDM_A3_R1_BATCH_1D","16"))
BATCH_2D=int(os.getenv("CIDM_A3_R1_BATCH_2D","4"))

TASKS={
    "advection_1d":{
        "family":"transport",
        "dim":1,"channels":1,"spatial":[256],
        "cfg":{"latent_dim":42,"hidden":72,"soc_modes1":1,"soc_modes2":4},
        "expected_params":500084,
    },
    "burgers_1d":{
        "family":"nonlinear_transport",
        "dim":1,"channels":1,"spatial":[256],
        "cfg":{"latent_dim":42,"hidden":72,"soc_modes1":1,"soc_modes2":4},
        "expected_params":500084,
    },
    "reacdiff_2d":{
        "family":"reaction_diffusion_2d",
        "dim":2,"channels":2,"spatial":[128,128],
        "cfg":{"latent_dim":20,"hidden":92,"soc_modes1":2,"soc_modes2":14},
        "expected_params":500084,
    },
    "swe_2d":{
        "family":"shallow_water",
        "dim":2,"channels":1,"spatial":[128,128],
        "cfg":{"latent_dim":22,"hidden":84,"soc_modes1":4,"soc_modes2":12},
        "expected_params":500038,
    },
    "cfd_2d":{
        "family":"compressible_cfd_2d",
        "dim":2,"channels":4,"spatial":[64,64],
        "cfg":{"latent_dim":46,"hidden":48,"soc_modes1":2,"soc_modes2":10},
        "expected_params":499968,
    },
}

MAIN_VARIANTS={
    "NoPhase":{"phase"},
    "NoSkew":{"skew"},
    "NoFlux":{"flux"},
    "NoHidden":{"hidden"},
    "NoSOC":{"soc"},
    "NoBER":{"ber"},
    "MinimalComplete":{
        "phase","skew","flux","hidden","soc","ber","spectral_readout"
    },
}
SUPP_VARIANTS={
    "NoSpectralReadout":{"spectral_readout"},
}
ALL_VARIANTS={**MAIN_VARIANTS,**SUPP_VARIANTS}

REPORT={
    "experiment":"CIDM Foundation A3-R1 Conditional Component Attribution",
    "started_at":time.strftime("%Y-%m-%d %H:%M:%S"),
    "tasks":TASKS,
    "main_variants":{k:sorted(v) for k,v in MAIN_VARIANTS.items()},
    "supp_variants":{k:sorted(v) for k,v in SUPP_VARIANTS.items()},
    "seeds":SEEDS,
}

print("GPU:",torch.cuda.get_device_name(0))
print("Formal main runs:",len(MAIN_VARIANTS)*len(TASKS)*len(SEEDS))
print("Supplement runs:",len(SUPP_VARIANTS)*len(TASKS)*len(SUPP_SEEDS))

def seed_all(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def portable_state_dict(model):
    raw=model.state_dict()
    out=OrderedDict()
    for k,v in raw.items():
        if k.split(".")[-1]=="_metadata":
            continue
        if torch.is_tensor(v):
            out[k]=v.detach().cpu().clone()
        else:
            out[k]=copy.deepcopy(v)
    if hasattr(raw,"_metadata"):
        out._metadata=copy.deepcopy(raw._metadata)
    return out

def load_portable_state_dict(model,sd):
    inc=model.load_state_dict(sd,strict=False)
    missing=list(inc.missing_keys)
    unexpected=[k for k in inc.unexpected_keys if k.split(".")[-1]!="_metadata"]
    if missing or unexpected:
        raise RuntimeError((missing,unexpected))

def save_json(obj,path):
    Path(path).write_text(
        json.dumps(obj,ensure_ascii=False,indent=2,default=str),
        encoding="utf-8"
    )

def sha256_file(path,chunk=8*1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(chunk),b""):
            h.update(b)
    return h.hexdigest()


## 1. Frozen archived CIDM-Complete source

In [ ]:

# ============================================================
# 1. Frozen archived source
# ============================================================
ARCHIVE_SOURCE='"""\nCIDM architecture archive\n=========================\n\nReference PyTorch implementation for the finalized model family:\n\n1. CIDM / Closed Information Dynamics Model\n   - General-purpose closed information dynamics backbone.\n   - Corresponds conceptually to the previous experimental `sticm_v3_full`.\n\n2. CIDM-BER / CIDM with Boundary Energy Reservoir\n   - Physics-closure enhanced variant.\n   - Adds a latent Spectral Operator Core and a Boundary Energy Reservoir (BER)\n     residual closure layer.\n   - Corresponds conceptually to the previous experimental `sticm_soc_ber`.\n\nThis file is intended for architecture archiving and future experiment reuse. It\ncontains only model definitions and lightweight diagnostics; training/evaluation\nloops are intentionally kept outside this file.\n\nExpected input format\n---------------------\n`history`: Tensor[B, L, C, H, W]\n`port`: optional Tensor[B, C_port, H, W] or Tensor[B, horizon, C_port, H, W]\n`target`: optional Tensor[B, horizon, C, H, W] for diagnostics/loss construction\n\nOutput dictionary\n-----------------\n`pred`: Tensor[B, horizon, C, H, W]\n`latent`: final latent state\n`latent_seq`: Tensor[B, horizon, D, H, W]\n`diagnostics`: scalar tensors for energy drift, SOC strength, BER strength, etc.\n\nAuthoring note\n--------------\nThis is a cleaned reference architecture. Exact Colab benchmark implementations\nmay contain additional data-generation, plotting, logging, and ablation utilities.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Dict, Optional, Tuple\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\n@dataclass\nclass CIDMConfig:\n    channels: int = 1\n    hist_len: int = 4\n    horizon: int = 6\n    latent_dim: int = 32\n    hidden: int = 64\n    port_channels: int = 1\n    use_phase_encoder: bool = True\n    use_time_encoder: bool = True\n    use_hidden_reservoir: bool = True\n    use_flux: bool = True\n    use_reverse_diagnostics: bool = True\n    # CIDM-BER options\n    soc_modes1: int = 8\n    soc_modes2: int = 8\n    ber_add_scale: float = 0.05\n    ber_enabled: bool = True\n    soc_enabled: bool = True\n\n\ndef count_parameters(model: nn.Module) -> int:\n    """Return the number of trainable parameters."""\n    return sum(p.numel() for p in model.parameters() if p.requires_grad)\n\n\ndef conv_block(in_ch: int, out_ch: int, hidden: int, groups: int = 1) -> nn.Sequential:\n    return nn.Sequential(\n        nn.Conv2d(in_ch, hidden, 3, padding=1),\n        nn.GroupNorm(groups, hidden),\n        nn.SiLU(),\n        nn.Conv2d(hidden, hidden, 3, padding=1),\n        nn.GroupNorm(groups, hidden),\n        nn.SiLU(),\n        nn.Conv2d(hidden, out_ch, 1),\n    )\n\n\nclass SpaceEncoder(nn.Module):\n    """Encode the latest observed spatial field into latent information state."""\n\n    def __init__(self, channels: int, latent_dim: int):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Conv2d(channels, latent_dim, 3, padding=1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, latent_dim, 3, padding=1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, latent_dim, 1),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.net(x)\n\n\nclass TimeHistoryEncoder(nn.Module):\n    """Encode a finite observation history as temporal context."""\n\n    def __init__(self, channels: int, hist_len: int, latent_dim: int):\n        super().__init__()\n        self.hist_len = hist_len\n        self.net = nn.Sequential(\n            nn.Conv2d(channels * hist_len, latent_dim, 3, padding=1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, latent_dim, 3, padding=1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, latent_dim, 1),\n        )\n\n    def forward(self, history: torch.Tensor) -> torch.Tensor:\n        # history: B, L, C, H, W\n        b, l, c, h, w = history.shape\n        if l != self.hist_len:\n            raise ValueError(f"Expected hist_len={self.hist_len}, got {l}.")\n        return self.net(history.reshape(b, l * c, h, w))\n\n\nclass PhaseSpectralEncoder(nn.Module):\n    """FFT-based phase/amplitude encoder projected back to the spatial latent grid."""\n\n    def __init__(self, channels: int, latent_dim: int):\n        super().__init__()\n        self.freq_net = nn.Sequential(\n            nn.Conv2d(channels * 3, latent_dim, 1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, latent_dim, 1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n        )\n        self.spatial_net = nn.Sequential(\n            nn.Conv2d(latent_dim, latent_dim, 3, padding=1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, latent_dim, 1),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        z = torch.fft.fft2(x, dim=(-2, -1), norm="ortho")\n        amp = torch.log1p(torch.abs(z))\n        feats = torch.cat([z.real, z.imag, amp], dim=1)\n        f = self.freq_net(feats)\n        # Return to the spatial support while retaining frequency-aware features.\n        f_complex = torch.complex(f, torch.zeros_like(f))\n        spatial = torch.fft.ifft2(f_complex, dim=(-2, -1), norm="ortho").real\n        return self.spatial_net(spatial)\n\n\nclass FieldDecoder(nn.Module):\n    """Decode latent information state to physical/observed field."""\n\n    def __init__(self, latent_dim: int, channels: int, hidden: int = 64):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Conv2d(latent_dim, hidden, 3, padding=1),\n            nn.GroupNorm(1, hidden),\n            nn.SiLU(),\n            nn.Conv2d(hidden, hidden, 3, padding=1),\n            nn.GroupNorm(1, hidden),\n            nn.SiLU(),\n            nn.Conv2d(hidden, channels, 1),\n        )\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        return self.net(psi)\n\n\nclass SpectralFieldDecoder(nn.Module):\n    """Small spectral residual decoder used by CIDM for phase-aware readout."""\n\n    def __init__(self, latent_dim: int, channels: int):\n        super().__init__()\n        self.mix = nn.Sequential(\n            nn.Conv2d(latent_dim, latent_dim, 1),\n            nn.SiLU(),\n            nn.Conv2d(latent_dim, channels, 1),\n        )\n        self.scale = nn.Parameter(torch.tensor(-2.2))\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        z = torch.fft.fft2(psi, dim=(-2, -1), norm="ortho")\n        # Simple low-pass emphasis for phase-stable readout.\n        H, W = psi.shape[-2:]\n        ky = torch.fft.fftfreq(H, device=psi.device).view(1, 1, H, 1)\n        kx = torch.fft.fftfreq(W, device=psi.device).view(1, 1, 1, W)\n        filt = torch.exp(-6.0 * (kx ** 2 + ky ** 2))\n        low = torch.fft.ifft2(z * filt, dim=(-2, -1), norm="ortho").real\n        return torch.sigmoid(self.scale) * self.mix(low)\n\n\nclass AntisymmetricFlux(nn.Module):\n    """Conservative local exchange field.\n\n    For each neighbor pair, the edge update is constructed as f(a,b)-f(b,a),\n    then added to one side and subtracted from the other. This encodes the idea\n    that local exchanges can redistribute information without creating it.\n    """\n\n    def __init__(self, latent_dim: int, hidden: int):\n        super().__init__()\n        self.edge_net = nn.Sequential(\n            nn.Conv2d(2 * latent_dim, hidden, 1),\n            nn.SiLU(),\n            nn.Conv2d(hidden, latent_dim, 1),\n        )\n\n    def edge(self, a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:\n        return self.edge_net(torch.cat([a, b], dim=1)) - self.edge_net(torch.cat([b, a], dim=1))\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        out = torch.zeros_like(psi)\n        # x direction\n        right = torch.roll(psi, shifts=-1, dims=-1)\n        fx = self.edge(psi, right)\n        out = out + fx - torch.roll(fx, shifts=1, dims=-1)\n        # y direction; safe for H=1 because roll is identity and antisymmetry cancels.\n        down = torch.roll(psi, shifts=-1, dims=-2)\n        fy = self.edge(psi, down)\n        out = out + fy - torch.roll(fy, shifts=1, dims=-2)\n        return out\n\n\nclass SkewChannelRotation(nn.Module):\n    """Learned skew-symmetric channel mixing for approximately conservative flow."""\n\n    def __init__(self, latent_dim: int):\n        super().__init__()\n        self.raw = nn.Parameter(torch.randn(latent_dim, latent_dim) * 0.02)\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        A = self.raw - self.raw.t()\n        return torch.einsum("ij,bjhw->bihw", A, psi)\n\n\nclass HiddenReservoirTransfer(nn.Module):\n    """Internal hidden reservoir transfer for unobserved latent exchange."""\n\n    def __init__(self, latent_dim: int, hidden: int):\n        super().__init__()\n        self.net = conv_block(latent_dim, latent_dim, hidden)\n        self.scale = nn.Parameter(torch.tensor(-2.0))\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        return torch.sigmoid(self.scale) * self.net(psi)\n\n\nclass ClosedInformationDynamicsCore(nn.Module):\n    """CIDM closed latent vector field."""\n\n    def __init__(self, latent_dim: int, hidden: int, use_flux: bool = True, use_hidden_reservoir: bool = True):\n        super().__init__()\n        self.use_flux = use_flux\n        self.use_hidden_reservoir = use_hidden_reservoir\n        self.skew = SkewChannelRotation(latent_dim)\n        self.flux = AntisymmetricFlux(latent_dim, hidden)\n        self.hidden = HiddenReservoirTransfer(latent_dim, hidden)\n        self.local = conv_block(latent_dim, latent_dim, hidden)\n        self.step_scale = nn.Parameter(torch.tensor(0.18))\n\n    def vector_field(self, psi: torch.Tensor) -> torch.Tensor:\n        dpsi = self.skew(psi)\n        if self.use_flux:\n            dpsi = dpsi + self.flux(psi)\n        if self.use_hidden_reservoir:\n            dpsi = dpsi + 0.25 * self.hidden(psi)\n        # A small local residual stabilizes non-Hamiltonian observed fields.\n        dpsi = dpsi + 0.10 * self.local(psi)\n        return dpsi\n\n    def forward(self, psi: torch.Tensor, dt_scale: float = 1.0) -> torch.Tensor:\n        alpha = torch.tanh(self.step_scale) * dt_scale\n        return psi + alpha * self.vector_field(psi)\n\n\nclass LatentSpectralOperatorCore(nn.Module):\n    """Latent-domain spectral operator for CIDM-BER.\n\n    This is a residual spectral vector field inside the latent dynamics, not an\n    external FNO baseline. It learns complex low-frequency mode mixing and maps\n    it back to the latent grid as a global correction to local closed dynamics.\n    """\n\n    def __init__(self, latent_dim: int, hidden: int = 64, modes1: int = 8, modes2: int = 8):\n        super().__init__()\n        self.latent_dim = latent_dim\n        self.modes1 = modes1\n        self.modes2 = modes2\n        scale = 1.0 / max(1, latent_dim)\n        self.weight_pos = nn.Parameter(scale * torch.randn(latent_dim, latent_dim, modes1, modes2, 2))\n        self.weight_neg = nn.Parameter(scale * torch.randn(latent_dim, latent_dim, modes1, modes2, 2))\n        self.pre = nn.Sequential(\n            nn.Conv2d(latent_dim, latent_dim, 1),\n            nn.GroupNorm(1, latent_dim),\n            nn.SiLU(),\n        )\n        self.post = nn.Sequential(\n            nn.Conv2d(latent_dim, hidden, 1),\n            nn.SiLU(),\n            nn.Conv2d(hidden, latent_dim, 1),\n        )\n        self.logit_scale = nn.Parameter(torch.tensor(-1.8))\n\n    @staticmethod\n    def compl_mul2d(x: torch.Tensor, wri: torch.Tensor) -> torch.Tensor:\n        w = torch.view_as_complex(wri.contiguous())\n        return torch.einsum("bihw,iohw->bohw", x, w)\n\n    def forward(self, psi: torch.Tensor) -> torch.Tensor:\n        B, D, H, W = psi.shape\n        x = self.pre(psi)\n        x_ft = torch.fft.rfft2(x, dim=(-2, -1), norm="ortho")\n        out_ft = torch.zeros(B, D, H, W // 2 + 1, dtype=torch.cfloat, device=psi.device)\n        m1 = min(self.modes1, H)\n        m2 = min(self.modes2, W // 2 + 1)\n        out_ft[:, :, :m1, :m2] = self.compl_mul2d(x_ft[:, :, :m1, :m2], self.weight_pos[:, :, :m1, :m2])\n        if H > 1:\n            out_ft[:, :, -m1:, :m2] = self.compl_mul2d(x_ft[:, :, -m1:, :m2], self.weight_neg[:, :, :m1, :m2])\n        y = torch.fft.irfft2(out_ft, s=(H, W), dim=(-2, -1), norm="ortho")\n        return torch.sigmoid(self.logit_scale) * self.post(y)\n\n\nclass BoundaryEnergyReservoir(nn.Module):\n    """Boundary Energy Reservoir (BER) residual closure layer.\n\n    BER is a virtual external energy reservoir, not a full external-state model.\n    It learns a minimal residual exchange field that accounts for open-system\n    effects not captured by the internal closed dynamics.\n    """\n\n    def __init__(self, latent_dim: int, port_channels: int, hidden: int):\n        super().__init__()\n        self.port_channels = max(1, port_channels)\n        in_ch = latent_dim * 2 + self.port_channels\n        self.net = nn.Sequential(\n            nn.Conv2d(in_ch, hidden, 3, padding=1),\n            nn.GroupNorm(1, hidden),\n            nn.SiLU(),\n            nn.Conv2d(hidden, hidden, 3, padding=1),\n            nn.GroupNorm(1, hidden),\n            nn.SiLU(),\n            nn.Conv2d(hidden, latent_dim, 3, padding=1),\n        )\n        mask_hidden = max(8, hidden // 2)\n        self.mask = nn.Sequential(\n            nn.Conv2d(in_ch, mask_hidden, 3, padding=1),\n            nn.SiLU(),\n            nn.Conv2d(mask_hidden, 1, 1),\n            nn.Sigmoid(),\n        )\n        self.logit_scale = nn.Parameter(torch.tensor(-3.0))\n\n    def _port_to_grid(self, psi: torch.Tensor, port: Optional[torch.Tensor]) -> torch.Tensor:\n        B, _, H, W = psi.shape\n        if port is None or port.numel() == 0:\n            return torch.zeros(B, self.port_channels, H, W, device=psi.device, dtype=psi.dtype)\n        if port.dim() != 4:\n            raise ValueError("port must be Tensor[B, C_port, H, W] for one step.")\n        p = port\n        if p.shape[-2:] != (H, W):\n            p = F.interpolate(p, size=(H, W), mode="bilinear", align_corners=False)\n        if p.shape[1] < self.port_channels:\n            pad = torch.zeros(B, self.port_channels - p.shape[1], H, W, device=psi.device, dtype=psi.dtype)\n            p = torch.cat([p, pad], dim=1)\n        elif p.shape[1] > self.port_channels:\n            p = p[:, : self.port_channels]\n        return p\n\n    def forward(\n        self,\n        psi: torch.Tensor,\n        internal_delta: torch.Tensor,\n        port: Optional[torch.Tensor] = None,\n    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        p = self._port_to_grid(psi, port)\n        x = torch.cat([psi, internal_delta, p], dim=1)\n        raw = torch.tanh(self.net(x))\n        mask = self.mask(x)\n        scale = torch.sigmoid(self.logit_scale)\n        ber_delta = scale * mask * raw\n        info = {\n            "ber_strength": ber_delta.abs().mean(),\n            "ber_mask_mean": mask.mean(),\n            "ber_scale": scale,\n        }\n        return ber_delta, info\n\n\nclass CIDM(nn.Module):\n    """Closed Information Dynamics Model.\n\n    General-purpose主体架构. This version corresponds to the stabilized closed\n    dynamics backbone: spatial information encoding, temporal history encoding,\n    phase-spectral feature encoding, conservative local exchange, internal hidden\n    reservoir transfer, and autoregressive latent rollout.\n    """\n\n    def __init__(self, config: CIDMConfig):\n        super().__init__()\n        self.config = config\n        D = config.latent_dim\n        H = config.hidden\n        self.space_encoder = SpaceEncoder(config.channels, D)\n        self.time_encoder = TimeHistoryEncoder(config.channels, config.hist_len, D)\n        self.phase_encoder = PhaseSpectralEncoder(config.channels, D)\n        fuse_inputs = D + (D if config.use_time_encoder else 0) + (D if config.use_phase_encoder else 0)\n        self.fuse = nn.Sequential(nn.Conv2d(fuse_inputs, D, 1), nn.SiLU(), nn.Conv2d(D, D, 1))\n        self.core = ClosedInformationDynamicsCore(D, H, use_flux=config.use_flux, use_hidden_reservoir=config.use_hidden_reservoir)\n        self.decoder = FieldDecoder(D, config.channels, H)\n        self.spectral_decoder = SpectralFieldDecoder(D, config.channels)\n\n    def encode(self, history: torch.Tensor) -> torch.Tensor:\n        last = history[:, -1]\n        feats = [self.space_encoder(last)]\n        if self.config.use_time_encoder:\n            feats.append(self.time_encoder(history))\n        if self.config.use_phase_encoder:\n            feats.append(self.phase_encoder(last))\n        return self.fuse(torch.cat(feats, dim=1))\n\n    def decode(self, psi: torch.Tensor) -> torch.Tensor:\n        return self.decoder(psi) + self.spectral_decoder(psi)\n\n    def step(self, psi: torch.Tensor, step_index: int = 0, port: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        psi_next = self.core(psi)\n        return psi_next, {"soc_strength": torch.zeros((), device=psi.device), "ber_strength": torch.zeros((), device=psi.device)}\n\n    def _select_port_step(self, port: Optional[torch.Tensor], t: int) -> Optional[torch.Tensor]:\n        if port is None:\n            return None\n        if port.dim() == 5:\n            return port[:, min(t, port.shape[1] - 1)]\n        return port\n\n    def forward(\n        self,\n        history: torch.Tensor,\n        horizon: Optional[int] = None,\n        port: Optional[torch.Tensor] = None,\n        target: Optional[torch.Tensor] = None,\n    ) -> Dict[str, torch.Tensor]:\n        horizon = int(horizon or self.config.horizon)\n        psi0 = self.encode(history)\n        psi = psi0\n        preds = []\n        latent_seq = []\n        soc_strengths, ber_strengths = [], []\n        energy0 = (psi0 ** 2).mean(dim=(1, 2, 3))\n        energies = []\n        for t in range(horizon):\n            psi, info = self.step(psi, t, self._select_port_step(port, t))\n            y = self.decode(psi)\n            preds.append(y)\n            latent_seq.append(psi)\n            energies.append((psi ** 2).mean(dim=(1, 2, 3)))\n            soc_strengths.append(info.get("soc_strength", torch.zeros((), device=psi.device)))\n            ber_strengths.append(info.get("ber_strength", torch.zeros((), device=psi.device)))\n        pred = torch.stack(preds, dim=1)\n        latent = torch.stack(latent_seq, dim=1)\n        energy = torch.stack(energies, dim=1)\n        diagnostics = {\n            "energy_drift": (energy[:, -1] - energy0).abs().mean(),\n            "soc_strength": torch.stack([s if s.dim() else s.expand(1) for s in soc_strengths]).mean(),\n            "ber_strength": torch.stack([b if b.dim() else b.expand(1) for b in ber_strengths]).mean(),\n        }\n        if target is not None:\n            diagnostics["rmse"] = torch.sqrt(F.mse_loss(pred, target))\n            diagnostics["mae"] = F.l1_loss(pred, target)\n        return {"pred": pred, "latent": psi, "latent_seq": latent, "diagnostics": diagnostics}\n\n\nclass CIDM_BER(CIDM):\n    """CIDM with Spectral Operator Core and Boundary Energy Reservoir.\n\n    Physics-enhanced variant for systems with salient wave/flow propagation,\n    quasi-closed dynamics, boundary exchange, or energy closure requirements.\n    """\n\n    def __init__(self, config: CIDMConfig):\n        super().__init__(config)\n        self.soc = LatentSpectralOperatorCore(\n            config.latent_dim,\n            hidden=config.hidden,\n            modes1=config.soc_modes1,\n            modes2=config.soc_modes2,\n        )\n        self.ber = BoundaryEnergyReservoir(config.latent_dim, config.port_channels, config.hidden)\n        self.ber_add_scale = float(config.ber_add_scale)\n        self.soc_enabled = bool(config.soc_enabled)\n        self.ber_enabled = bool(config.ber_enabled)\n\n    def step(self, psi: torch.Tensor, step_index: int = 0, port: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:\n        psi_int = self.core(psi)\n        soc_delta = self.soc(psi) if self.soc_enabled else torch.zeros_like(psi)\n        psi_int = psi_int + soc_delta\n        internal_delta = psi_int - psi\n        if self.ber_enabled:\n            ber_delta, ber_info = self.ber(psi, internal_delta.detach(), port)\n            psi_next = psi_int + self.ber_add_scale * ber_delta\n        else:\n            ber_delta = torch.zeros_like(psi)\n            ber_info = {"ber_strength": torch.zeros((), device=psi.device), "ber_mask_mean": torch.zeros((), device=psi.device)}\n            psi_next = psi_int\n        info = {\n            "soc_strength": soc_delta.abs().mean(),\n            "ber_strength": ber_info["ber_strength"],\n            "ber_mask_mean": ber_info.get("ber_mask_mean", torch.zeros((), device=psi.device)),\n            "ber_budget_proxy": (ber_delta.mean(dim=(1, 2, 3)) - internal_delta.mean(dim=(1, 2, 3))).abs().mean(),\n        }\n        return psi_next, info\n\n\ndef build_cidm(config: Optional[CIDMConfig] = None) -> CIDM:\n    return CIDM(config or CIDMConfig())\n\n\ndef build_cidm_ber(config: Optional[CIDMConfig] = None) -> CIDM_BER:\n    cfg = config or CIDMConfig()\n    return CIDM_BER(cfg)\n\n\n# Backward-compatible aliases for previous internal experiment names.\nClosedInformationDynamicsModel = CIDM\nClosedInformationDynamicsBERModel = CIDM_BER\nSTICM_V3_FULL_ALIAS = CIDM\nSTICM_SOC_BER_ALIAS = CIDM_BER\n\n\nif __name__ == "__main__":\n    torch.manual_seed(0)\n    cfg = CIDMConfig(channels=1, hist_len=4, horizon=1, latent_dim=4, hidden=8, port_channels=1)\n    x = torch.randn(1, cfg.hist_len, cfg.channels, 4, 4)\n    port = torch.randn(1, cfg.port_channels, 4, 4)\n    for cls in [CIDM, CIDM_BER]:\n        model = cls(cfg)\n        out = model(x, port=port)\n        print(cls.__name__, out["pred"].shape, "params=", count_parameters(model), out["diagnostics"])\n'
EXPECTED_ARCHIVE_SHA256="f9b1cf4ff7630f5540bb3adde20e4c4b81fff3218d9325ea3105bf28b1d97665"

arch_file=FROZEN/"cidm_architectures.py"
arch_file.write_text(ARCHIVE_SOURCE,encoding="utf-8")
got=sha256_file(arch_file)
print("CIDM archive SHA256:",got)
assert got==EXPECTED_ARCHIVE_SHA256

import importlib.util
spec=importlib.util.spec_from_file_location("cidm_a3_archive",arch_file)
cidm_mod=importlib.util.module_from_spec(spec)
sys.modules["cidm_a3_archive"]=cidm_mod
spec.loader.exec_module(cidm_mod)

CIDMConfig=cidm_mod.CIDMConfig
CIDM_BER=cidm_mod.CIDM_BER
REPORT["archive_sha256"]=got



## 2. Parameter-preserving attribution model

不重写任何 branch 内部算法，只控制 branch 输出是否进入计算图。

`FullInstrumented(ablate=∅)` 必须与 archived `CIDM_BER` 完全等价。


In [ ]:

# ============================================================
# 2. Parameter-preserving masked ablation implementation
# ============================================================
class CIDMCompleteAttribution(CIDM_BER):
    def __init__(self,config,ablate=()):
        super().__init__(config)
        self.ablate=frozenset(ablate)

    def encode(self,history):
        last=history[:,-1]
        zs=self.space_encoder(last)
        zt=self.time_encoder(history)
        zp=self.phase_encoder(last)
        if "phase" in self.ablate:
            zp=torch.zeros_like(zp)
        return self.fuse(torch.cat([zs,zt,zp],dim=1))

    def decode(self,psi):
        y=self.decoder(psi)
        if "spectral_readout" not in self.ablate:
            y=y+self.spectral_decoder(psi)
        return y

    def step(self,psi,step_index=0,port=None):
        zero=torch.zeros_like(psi)

        skew=self.core.skew(psi) if "skew" not in self.ablate else zero
        flux=self.core.flux(psi) if "flux" not in self.ablate else zero
        hidden=self.core.hidden(psi) if "hidden" not in self.ablate else zero
        local=self.core.local(psi)  # Core mechanism: always retained.

        dpsi=skew+flux+0.25*hidden+0.10*local
        alpha=torch.tanh(self.core.step_scale)
        psi_int=psi+alpha*dpsi

        soc_delta=self.soc(psi) if "soc" not in self.ablate else zero
        psi_int=psi_int+soc_delta
        internal_delta=psi_int-psi

        if "ber" not in self.ablate:
            ber_delta,ber_info=self.ber(
                psi,internal_delta.detach(),port
            )
            psi_next=psi_int+self.ber_add_scale*ber_delta
        else:
            ber_delta=zero
            ber_info={
                "ber_strength":torch.zeros((),device=psi.device),
                "ber_mask_mean":torch.zeros((),device=psi.device),
            }
            psi_next=psi_int

        return psi_next,{
            "skew_strength":skew.abs().mean(),
            "flux_strength":flux.abs().mean(),
            "hidden_strength":hidden.abs().mean(),
            "local_strength":local.abs().mean(),
            "soc_strength":soc_delta.abs().mean(),
            "ber_strength":ber_info["ber_strength"],
        }

class AttributionWrapper(nn.Module):
    def __init__(self,task,variant,ablate):
        super().__init__()
        meta=TASKS[task]
        cfgd=meta["cfg"]
        self.dim=meta["dim"]
        cfg=CIDMConfig(
            channels=meta["channels"],
            hist_len=HISTORY,
            horizon=HORIZON,
            latent_dim=cfgd["latent_dim"],
            hidden=cfgd["hidden"],
            port_channels=1,
            use_phase_encoder=True,
            use_time_encoder=True,
            use_hidden_reservoir=True,
            use_flux=True,
            use_reverse_diagnostics=True,
            soc_modes1=cfgd["soc_modes1"],
            soc_modes2=cfgd["soc_modes2"],
            ber_add_scale=0.05,
            ber_enabled=True,
            soc_enabled=True,
        )
        self.variant=variant
        self.model=CIDMCompleteAttribution(cfg,ablate=ablate)

    def _lift(self,h):
        return h.unsqueeze(-2) if self.dim==1 else h

    def rollout(self,h,K):
        out=self.model(self._lift(h),horizon=K,port=None)
        p=out["pred"]
        return p.squeeze(-2) if self.dim==1 else p

def build_variant(task,variant):
    if variant=="FullInstrumented":
        ablate=set()
    else:
        ablate=ALL_VARIANTS[variant]
    return AttributionWrapper(task,variant,ablate)

# Nominal parameter count must remain identical to archived Complete.
param_rows=[]
for task,meta in TASKS.items():
    for variant in ["FullInstrumented"]+list(ALL_VARIANTS):
        m=build_variant(task,variant)
        p=count_params(m)
        ok=(p==meta["expected_params"])
        param_rows.append({
            "task":task,"variant":variant,"params":p,
            "expected":meta["expected_params"],"equal":ok
        })
        del m
param_df=pd.DataFrame(param_rows)
display(param_df)
assert param_df["equal"].all()
param_df.to_csv(OUT/"A3_R1_nominal_parameter_audit.csv",index=False)


## 3. HF Dataset: restore exact A2 caches + Full/Core references + A3 progress

In [ ]:

# ============================================================
# 3. HF Dataset persistent runtime
# ============================================================
HF_DATASET_REPO_ID=os.getenv(
    "CIDM_HF_DATASET_REPO","wuff-mann/CIDM-v1"
).strip()
HF_BASE_PREFIX="A2_BroadDynamics_500K_3Seed"
HF_PREFIX="A3_R1_ComponentAttribution_500K_3Seed"
HF_CHECKPOINT_EVERY=int(os.getenv("CIDM_HF_CHECKPOINT_EVERY","500"))
HF_MAX_RETRIES=int(os.getenv("CIDM_HF_MAX_RETRIES","4"))

try:
    from google.colab import userdata
    HF_TOKEN=userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN=os.getenv("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is required for A3-R1.")

subprocess.run(
    [sys.executable,"-m","pip","install","-q","-U","huggingface_hub"],
    check=True
)
from huggingface_hub import HfApi,hf_hub_download,CommitOperationAdd
HF_API=HfApi(token=HF_TOKEN)

def retry(fn,*args,**kwargs):
    last=None
    for a in range(1,HF_MAX_RETRIES+1):
        try:
            return fn(*args,**kwargs)
        except Exception as e:
            last=e
            if a==HF_MAX_RETRIES: break
            time.sleep(min(30,2**a))
    raise last

retry(HF_API.dataset_info,repo_id=HF_DATASET_REPO_ID,revision="main")
REMOTE=retry(
    HF_API.list_repo_files,
    repo_id=HF_DATASET_REPO_ID,
    repo_type="dataset",
    revision="main",
)
print("[HF DATASET VERIFIED]",HF_DATASET_REPO_ID)

def hfdl(remote):
    return Path(retry(
        hf_hub_download,
        repo_id=HF_DATASET_REPO_ID,
        filename=remote,
        repo_type="dataset",
        revision="main",
        token=HF_TOKEN,
        local_dir="/content/_cidm_a3_restore",
    ))

# -------- restore exact selected A2 compact caches --------
CACHE_FILES={}
for task in TASKS:
    cand=[
        f for f in REMOTE
        if f.startswith(HF_BASE_PREFIX+"/cache/")
        and Path(f).name.startswith(task+"_")
        and f.endswith(".pt")
    ]
    if len(cand)!=1:
        raise RuntimeError(
            f"Expected exactly one Base-A2 compact cache for {task}, got {cand}"
        )
    src=hfdl(cand[0])
    dst=CACHE_ROOT/Path(cand[0]).name
    if not dst.exists():
        shutil.copy2(src,dst)
    CACHE_FILES[task]=dst
    print("[A2 cache]",task,dst.name)

# -------- restore Base A2 Full Complete + Core references --------
REF_ROWS=[]
for task in TASKS:
    for seed in SEEDS:
        for model in ["CIDM-Complete","CIDM-Foundation-v1"]:
            name=f"A2__{model}__{task}__s{seed}_result.json"
            cand=[
                f for f in REMOTE
                if f.startswith(HF_BASE_PREFIX+"/")
                and Path(f).name==name
            ]
            if len(cand)!=1:
                raise RuntimeError(f"Missing exact Base-A2 reference {name}: {cand}")
            src=hfdl(cand[0])
            dst=BASE_REF/name
            if not dst.exists():
                shutil.copy2(src,dst)
            row=json.loads(dst.read_text(encoding="utf-8"))
            if row.get("status")!="COMPLETED":
                raise RuntimeError(f"Reference not completed: {name}")
            REF_ROWS.append(row)

ref_df=pd.DataFrame(REF_ROWS)
assert len(ref_df)==len(TASKS)*len(SEEDS)*2
ref_df.to_csv(OUT/"A3_R1_A2_reference_results.csv",index=False)

FULL_REF={
    (r["task"],int(r["seed"])):r
    for r in REF_ROWS if r["model"]=="CIDM-Complete"
}
CORE_REF={
    (r["task"],int(r["seed"])):r
    for r in REF_ROWS if r["model"]=="CIDM-Foundation-v1"
}

# -------- A3 progress restore --------
def scan_progress():
    rows=[]
    for p in OUT.glob("A3R1__*_result.json"):
        try:
            rows.append(json.loads(p.read_text(encoding="utf-8")))
        except Exception:
            pass
    return {
        "main_expected":len(MAIN_VARIANTS)*len(TASKS)*len(SEEDS),
        "supp_expected":len(SUPP_VARIANTS)*len(TASKS)*len(SUPP_SEEDS),
        "completed":sum(r.get("status")=="COMPLETED" for r in rows),
        "result_files":len(rows),
        "checkpoints":len(list(CKPT.glob("A3R1__*_latest.pt"))),
    }

def manifest(run_id=None,step=None,event=None):
    x=scan_progress()
    x.update({
        "repo_id":HF_DATASET_REPO_ID,
        "prefix":HF_PREFIX,
        "updated_at":datetime.now(timezone.utc).isoformat(),
        "current_run":run_id,
        "current_step":step,
        "event":event,
    })
    p=OUT/"A3_R1_progress_manifest.json"
    p.write_text(json.dumps(x,ensure_ascii=False,indent=2),encoding="utf-8")
    return p,x

def repo_path(p):
    return f"{HF_PREFIX}/{Path(p).resolve().relative_to(OUT.resolve()).as_posix()}"

def commit_files(files,msg):
    ops=[]
    for p in files:
        p=Path(p)
        if p.exists() and p.is_file():
            ops.append(CommitOperationAdd(
                path_in_repo=repo_path(p),
                path_or_fileobj=str(p)
            ))
    if ops:
        retry(
            HF_API.create_commit,
            repo_id=HF_DATASET_REPO_ID,
            repo_type="dataset",
            operations=ops,
            commit_message=msg,
        )

def sync_checkpoint(path,run_id,step):
    man,_=manifest(run_id,step,"MID_RUN_CHECKPOINT")
    try:
        commit_files([path,man],f"[A3-R1] checkpoint {run_id} step={step}")
        print("[HF checkpoint]",run_id,step)
    except Exception as e:
        print("[HF checkpoint warning]",e)

def sync_finished(run_id,result_path,history_path,latest_path):
    row=json.loads(Path(result_path).read_text(encoding="utf-8"))
    man,state=manifest(run_id,None,f"RUN_{row.get('status')}")
    partial=[]
    for q in sorted(OUT.glob("A3R1__*_result.json")):
        try: partial.append(json.loads(q.read_text(encoding="utf-8")))
        except Exception: pass
    pp=OUT/"A3_R1_run_results_partial.csv"
    pd.DataFrame(partial).to_csv(pp,index=False)
    files=[result_path,history_path,man,pp]
    try:
        commit_files(
            files,
            f"[A3-R1] {row.get('status')} {run_id} completed={state['completed']}"
        )
        if row.get("status")=="COMPLETED":
            try:
                retry(
                    HF_API.delete_file,
                    path_in_repo=repo_path(latest_path),
                    repo_id=HF_DATASET_REPO_ID,
                    repo_type="dataset",
                    commit_message=f"[A3-R1] cleanup {run_id} checkpoint"
                )
            except Exception:
                pass
    except Exception as e:
        print("[HF run sync warning]",e)

# Restore existing A3 artifacts.
for rf in [f for f in REMOTE if f.startswith(HF_PREFIX+"/")]:
    name=Path(rf).name
    if name.startswith("A3R1__") and name.endswith("_result.json"):
        dst=OUT/name
    elif name.startswith("A3R1__") and name.endswith("_history.csv"):
        dst=OUT/name
    elif name.startswith("A3R1__") and name.endswith("_latest.pt"):
        dst=CKPT/name
    elif name=="A3_R1_progress_manifest.json":
        dst=OUT/name
    else:
        continue
    try:
        src=hfdl(rf)
        dst.parent.mkdir(parents=True,exist_ok=True)
        if dst.exists() and dst.name.endswith("_result.json"):
            old=json.loads(dst.read_text(encoding="utf-8"))
            if old.get("status")=="COMPLETED":
                continue
        if dst.exists() and dst.name.endswith("_latest.pt"):
            l=torch.load(dst,map_location="cpu",weights_only=False)
            r=torch.load(src,map_location="cpu",weights_only=False)
            if int(l.get("step",-1))>=int(r.get("step",-1)):
                continue
        shutil.copy2(src,dst)
    except Exception as e:
        print("[restore warning]",rf,e)

print("[A3 restore]",scan_progress())


## 4. Exact cache contract + loaders

In [ ]:

# ============================================================
# 4. Exact A2 cache contract
# ============================================================
CACHES={}
for task,path in CACHE_FILES.items():
    c=torch.load(path,map_location="cpu",weights_only=False)
    h=c["splits"]["train"]["history"]
    meta=TASKS[task]
    dim=h.ndim-3
    channels=h.shape[2]
    spatial=list(h.shape[3:])
    assert dim==meta["dim"],(task,dim,meta["dim"])
    assert channels==meta["channels"],(task,channels,meta["channels"])
    assert spatial==meta["spatial"],(task,spatial,meta["spatial"])
    CACHES[task]=c

class Normalizer:
    def __init__(self,mean,std):
        self.mean=mean.float()
        self.std=std.float().clamp_min(1e-6)
    def denorm_target(self,x):
        shape=[1,1,-1]+[1]*(x.ndim-3)
        return x*self.std.view(*shape).to(x.device)+self.mean.view(*shape).to(x.device)

NORMALIZERS={
    t:Normalizer(c["norm"]["mean"],c["norm"]["std"])
    for t,c in CACHES.items()
}

class FrozenWindows(Dataset):
    def __init__(self,c,split,norm):
        self.h=c["splits"][split]["history"].float()
        self.y=c["splits"][split]["target"].float()
        self.norm=norm
    def __len__(self): return len(self.h)
    def __getitem__(self,i):
        h=self.h[i]; y=self.y[i]
        mh=self.norm.mean.view(1,-1,*([1]*(h.ndim-2)))
        sh=self.norm.std.view(1,-1,*([1]*(h.ndim-2)))
        return (h-mh)/sh,(y-mh)/sh

def loaders_for(task,seed):
    batch=BATCH_1D if TASKS[task]["dim"]==1 else BATCH_2D
    c=CACHES[task]; n=NORMALIZERS[task]
    g=torch.Generator().manual_seed(seed)
    return (
        DataLoader(FrozenWindows(c,"train",n),batch_size=batch,shuffle=True,
                   generator=g,num_workers=0,drop_last=True,pin_memory=True),
        DataLoader(FrozenWindows(c,"val",n),batch_size=batch,shuffle=False,
                   num_workers=0,pin_memory=True),
        DataLoader(FrozenWindows(c,"test",n),batch_size=batch,shuffle=False,
                   num_workers=0,pin_memory=True),
    )

cache_rows=[]
for task,c in CACHES.items():
    h=c["splits"]["train"]["history"]
    cache_rows.append({
        "task":task,"dim":h.ndim-3,"channels":h.shape[2],
        "spatial":list(h.shape[3:]),
        "train_windows":len(h),
        "val_windows":len(c["splits"]["val"]["history"]),
        "test_windows":len(c["splits"]["test"]["history"]),
    })
cache_df=pd.DataFrame(cache_rows)
cache_df.to_csv(OUT/"A3_R1_cache_contract.csv",index=False)
display(cache_df)


## 5. Metrics

In [ ]:

# ============================================================
# 5. Metrics
# ============================================================
def spectral_masks(shape,device):
    if len(shape)==1:
        f=torch.fft.rfftfreq(shape[0],device=device)
        q=f/f.max().clamp_min(1e-12)
    else:
        fy=torch.fft.fftfreq(shape[-2],device=device)[:,None]
        fx=torch.fft.rfftfreq(shape[-1],device=device)[None,:]
        q=torch.sqrt(fx*fx+fy*fy)
        q=q/q.max().clamp_min(1e-12)
    return {
        "low":q<=1/3,
        "mid":(q>1/3)&(q<=2/3),
        "high":q>2/3,
    }

def spectral_metrics(pred,target):
    dim=pred.ndim-3
    if dim==1:
        P=torch.fft.rfft(pred.float(),dim=-1,norm="ortho").abs()
        T=torch.fft.rfft(target.float(),dim=-1,norm="ortho").abs()
        masks=spectral_masks((pred.shape[-1],),pred.device)
    else:
        P=torch.fft.rfft2(pred.float(),dim=(-2,-1),norm="ortho").abs()
        T=torch.fft.rfft2(target.float(),dim=(-2,-1),norm="ortho").abs()
        masks=spectral_masks((pred.shape[-2],pred.shape[-1]),pred.device)
    out={}
    for band,m in masks.items():
        out[f"logspec_rmse_{band}"]=float(
            ((torch.log1p(P[...,m])-torch.log1p(T[...,m]))**2).mean().sqrt()
        )
    return out

@torch.no_grad()
def evaluate(model,loader,task):
    model.eval()
    norm=NORMALIZERS[task]
    mse=[]; mae=[]; rel=[]; leads=[]; specs=[]
    for h,y_n in loader:
        h=h.to(DEVICE); y_n=y_n.to(DEVICE)
        p_n=model.rollout(h,HORIZON)
        p=norm.denorm_target(p_n)
        y=norm.denorm_target(y_n)
        e=p-y
        mse.append(float((e**2).mean()))
        mae.append(float(e.abs().mean()))
        rel.append(float(
            torch.linalg.vector_norm(e.reshape(e.shape[0],-1),dim=1).div(
                torch.linalg.vector_norm(y.reshape(y.shape[0],-1),dim=1).clamp_min(1e-8)
            ).mean()
        ))
        leads.append(torch.sqrt((e**2).flatten(2).mean(-1)).mean(0).cpu())
        specs.append(spectral_metrics(p,y))
    le=torch.stack(leads).mean(0)
    out={
        "rmse":math.sqrt(np.mean(mse)),
        "mae":float(np.mean(mae)),
        "relative_l2":float(np.mean(rel)),
        "lead_rmse":[float(x) for x in le],
        "rollout_growth":float(le[-1]/le[0].clamp_min(1e-8)),
    }
    for k in specs[0]:
        out[k]=float(np.mean([s[k] for s in specs]))
    return out

@torch.no_grad()
def latency(model,loader,repeats=40,warmup=10):
    model.eval()
    h,_=next(iter(loader))
    h=h[:1].to(DEVICE)
    for _ in range(warmup): model.rollout(h,HORIZON)
    torch.cuda.synchronize()
    ts=[]
    for _ in range(repeats):
        torch.cuda.synchronize()
        t=time.perf_counter()
        model.rollout(h,HORIZON)
        torch.cuda.synchronize()
        ts.append((time.perf_counter()-t)*1000)
    return float(np.median(ts))



## 6. Q0/Q1/Q2 Qualification

正式消融之前先证明：
- instrumentation 不改变 Full；
- bypass 只切断目标 branch；
- nominal params 完全相同；
- 所有 40 个 task×variant 可以短训练。


In [ ]:

# ============================================================
# 6. Qualification
# ============================================================
qual=[]

# Q0: exact Full equivalence.
for task in TASKS:
    meta=TASKS[task]
    cfgd=meta["cfg"]
    cfg=CIDMConfig(
        channels=meta["channels"],hist_len=HISTORY,horizon=HORIZON,
        latent_dim=cfgd["latent_dim"],hidden=cfgd["hidden"],port_channels=1,
        use_phase_encoder=True,use_time_encoder=True,
        use_hidden_reservoir=True,use_flux=True,
        soc_modes1=cfgd["soc_modes1"],soc_modes2=cfgd["soc_modes2"],
        ber_add_scale=0.05,ber_enabled=True,soc_enabled=True,
    )
    seed_all(12345)
    original=CIDM_BER(cfg).to(DEVICE).float()
    instrumented=build_variant(task,"FullInstrumented").to(DEVICE).float()
    instrumented.model.load_state_dict(original.state_dict(),strict=True)

    _,va,_=loaders_for(task,17)
    h,_=next(iter(va))
    h=h[:2].to(DEVICE)
    lifted=h.unsqueeze(-2) if meta["dim"]==1 else h
    with torch.no_grad():
        p0=original(lifted,horizon=HORIZON,port=None)["pred"]
        p1=instrumented.model(lifted,horizon=HORIZON,port=None)["pred"]
    md=float((p0-p1).abs().max())
    ok=md<1e-6 and count_params(original)==count_params(instrumented)
    qual.append({
        "stage":"Q0_full_equivalence","task":task,
        "variant":"FullInstrumented","pass":ok,"max_abs_diff":md
    })
    del original,instrumented
    torch.cuda.empty_cache()

# Q1/Q2 for 8 variants × 5 tasks.
PREFIX={
    "phase":"model.phase_encoder.",
    "skew":"model.core.skew.",
    "flux":"model.core.flux.",
    "hidden":"model.core.hidden.",
    "soc":"model.soc.",
    "ber":"model.ber.",
    "spectral_readout":"model.spectral_decoder.",
}
for task in TASKS:
    tr,_,_=loaders_for(task,17)
    h,y=next(iter(tr))
    h=h.to(DEVICE); y=y.to(DEVICE)

    for variant,ablate in ALL_VARIANTS.items():
        seed_all(17)
        m=build_variant(task,variant).to(DEVICE).float()
        opt=torch.optim.AdamW(m.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)

        # One backward for branch isolation.
        opt.zero_grad(set_to_none=True)
        p=m.rollout(h,HORIZON)
        loss=F.mse_loss(p,y)
        loss.backward()

        finite=bool(torch.isfinite(loss))
        local_grad=0.0
        branch_grad={k:0.0 for k in PREFIX}
        for n,p0 in m.named_parameters():
            if p0.grad is None: continue
            g=float(p0.grad.detach().abs().sum())
            if n.startswith("model.core.local."):
                local_grad+=g
            for k,pref in PREFIX.items():
                if n.startswith(pref):
                    branch_grad[k]+=g

        isolation=True
        for comp in ablate:
            isolation=isolation and branch_grad[comp] <= 1e-12
        isolation=isolation and local_grad>1e-12

        # 30-step micro fit.
        initial=float(loss.detach())
        for s in range(MICRO_STEPS):
            opt.zero_grad(set_to_none=True)
            pp=m.rollout(h,HORIZON)
            ll=F.mse_loss(pp,y)
            if not torch.isfinite(ll):
                finite=False
                break
            ll.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(),1.0)
            opt.step()
        final=float(ll.detach()) if finite else float("inf")
        stable=finite and final < max(initial*1.5,initial+1e-8)

        # checkpoint round trip
        tmp=CKPT/f"_qual_{variant}_{task}.pt"
        torch.save({"model":portable_state_dict(m)},tmp)
        m2=build_variant(task,variant).to(DEVICE).float()
        load_portable_state_dict(
            m2,torch.load(tmp,map_location="cpu",weights_only=False)["model"]
        )
        tmp.unlink(missing_ok=True)

        ok=isolation and stable
        qual.append({
            "stage":"Q1Q2_branch_micro",
            "task":task,"variant":variant,
            "pass":ok,
            "initial_loss":initial,"final_loss":final,
            "local_grad":local_grad,
            "isolation":isolation,
            **{f"grad_{k}":v for k,v in branch_grad.items()},
        })

        del m,m2
        torch.cuda.empty_cache()
        gc.collect()

qual_df=pd.DataFrame(qual)
qual_df.to_csv(OUT/"A3_R1_qualification.csv",index=False)
display(qual_df)
if not bool(qual_df["pass"].all()):
    raise RuntimeError("A3-R1 qualification failed; formal training blocked.")
print("A3-R1 QUALIFICATION: PASS")


## 7. Formal masked-ablation training

In [ ]:

# ============================================================
# 7. Formal training
# ============================================================
def normalized_tensors(task,split):
    c=CACHES[task]; n=NORMALIZERS[task]
    h=c["splits"][split]["history"].float()
    y=c["splits"][split]["target"].float()
    mh=n.mean.view(1,1,-1,*([1]*(h.ndim-3)))
    sh=n.std.view(1,1,-1,*([1]*(h.ndim-3)))
    return ((h-mh)/sh).contiguous(),((y-mh)/sh).contiguous()

TRAIN={t:normalized_tensors(t,"train") for t in TASKS}

def batch_plan(n,batch,seed,steps):
    rng=np.random.default_rng(seed)
    out=[]
    while len(out)<steps:
        perm=rng.permutation(n)
        usable=(len(perm)//batch)*batch
        for i in range(0,usable,batch):
            out.append(perm[i:i+batch].astype(np.int64))
            if len(out)>=steps: break
    return out

def train_variant(model,task,va,seed,run_id,latest):
    model=model.to(DEVICE).float()
    opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(
        opt,T_max=MAX_UPDATES,eta_min=LR*0.1
    )
    th,ty=TRAIN[task]
    batch=BATCH_1D if TASKS[task]["dim"]==1 else BATCH_2D
    plan=batch_plan(len(th),batch,seed,MAX_UPDATES)

    start=0; best=float("inf"); best_step=0; best_sd=None
    cref=float("inf"); bad=0; hist=[]; elapsed0=0.0
    if latest.exists():
        ck=torch.load(latest,map_location="cpu",weights_only=False)
        load_portable_state_dict(model,ck["model"])
        opt.load_state_dict(ck["optimizer"])
        sched.load_state_dict(ck["scheduler"])
        start=int(ck["step"]); best=float(ck["best"]); best_step=int(ck["best_step"])
        best_sd=ck.get("best_model"); cref=float(ck.get("cref",best))
        bad=int(ck.get("bad",0)); hist=list(ck.get("history",[]))
        elapsed0=float(ck.get("elapsed",0.0))
        print("[resume]",run_id,start)

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    t0=time.perf_counter()
    stop_reason="MAX_UPDATES"

    for s in range(start+1,MAX_UPDATES+1):
        idx=torch.as_tensor(plan[s-1],dtype=torch.long)
        h=th.index_select(0,idx).to(DEVICE,non_blocking=True)
        y=ty.index_select(0,idx).to(DEVICE,non_blocking=True)
        model.train(); opt.zero_grad(set_to_none=True)
        p=model.rollout(h,HORIZON)
        loss=F.mse_loss(p,y)
        if not torch.isfinite(loss):
            raise FloatingPointError((run_id,s,"loss"))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        opt.step(); sched.step()

        should_stop=False
        if s==1 or s%EVAL_EVERY==0 or s==MAX_UPDATES:
            vr=evaluate(model,va,task)["rmse"]
            if vr<best:
                best=vr; best_step=s; best_sd=portable_state_dict(model)
            if not math.isfinite(cref):
                cref=vr; bad=0
            elif vr<cref*(1-REL_MIN_DELTA):
                cref=vr; bad=0
            elif s>=MIN_UPDATES:
                bad+=1
            hist.append({
                "step":s,"train_mse":float(loss.detach()),
                "val_rmse":float(vr),"best_val_rmse":float(best),
                "best_step":best_step,"bad_evals":bad,
                "lr":float(opt.param_groups[0]["lr"]),
            })
            print(
                f"[{run_id}] {s}/{MAX_UPDATES} "
                f"val={vr:.6g} best={best:.6g}@{best_step} bad={bad}/{PATIENCE_EVALS}"
            )
            if s>=MIN_UPDATES and bad>=PATIENCE_EVALS:
                should_stop=True
                stop_reason="CONVERGED_PATIENCE"

        if s%100==0 or s==MAX_UPDATES or should_stop:
            torch.cuda.synchronize()
            elapsed=elapsed0+(time.perf_counter()-t0)
            torch.save({
                "step":s,"model":portable_state_dict(model),
                "optimizer":opt.state_dict(),"scheduler":sched.state_dict(),
                "best":best,"best_step":best_step,"best_model":best_sd,
                "cref":cref,"bad":bad,"history":hist,"elapsed":elapsed,
            },latest)
            if s%HF_CHECKPOINT_EVERY==0 or s==MAX_UPDATES or should_stop:
                sync_checkpoint(latest,run_id,s)
        if should_stop: break

    torch.cuda.synchronize()
    elapsed=elapsed0+(time.perf_counter()-t0)
    peak=int(torch.cuda.max_memory_allocated())
    actual=hist[-1]["step"] if hist else start
    if best_sd is None:
        best=evaluate(model,va,task)["rmse"]
        best_step=actual
        best_sd=portable_state_dict(model)
    load_portable_state_dict(model,best_sd)
    return model,{
        "best_val_rmse":best,"best_val_step":best_step,
        "actual_updates":actual,"stop_reason":stop_reason,
        "train_seconds":elapsed,"peak_vram_bytes":peak,
        "history":hist,
    }

PLAN=[]
for variant in MAIN_VARIANTS:
    for task in TASKS:
        for seed in SEEDS:
            PLAN.append((variant,task,seed,"MAIN"))
for variant in SUPP_VARIANTS:
    for task in TASKS:
        for seed in SUPP_SEEDS:
            PLAN.append((variant,task,seed,"SUPP"))

for variant,task,seed,tier in PLAN:
    run_id=f"A3R1__{variant}__{task}__s{seed}"
    rp=OUT/f"{run_id}_result.json"
    hp=OUT/f"{run_id}_history.csv"
    cp=CKPT/f"{run_id}_latest.pt"

    if rp.exists():
        try: old=json.loads(rp.read_text(encoding="utf-8"))
        except Exception: old={}
        if old.get("status")=="COMPLETED":
            print("[skip]",run_id)
            continue

    seed_all(seed)
    m=build_variant(task,variant)
    _,va,te=loaders_for(task,seed)

    try:
        m,ti=train_variant(m,task,va,seed,run_id,cp)
        pd.DataFrame(ti["history"]).to_csv(hp,index=False)
        ev=evaluate(m,te,task)
        lat=latency(m,te)
        full=FULL_REF[(task,seed)]
        core=CORE_REF[(task,seed)]

        row={
            "status":"COMPLETED","tier":tier,
            "variant":variant,"ablated":sorted(ALL_VARIANTS[variant]),
            "task":task,"family":TASKS[task]["family"],"seed":seed,
            "params":count_params(m),"lr":LR,
            "best_val_rmse":ti["best_val_rmse"],
            "best_val_step":ti["best_val_step"],
            "actual_updates":ti["actual_updates"],
            "stop_reason":ti["stop_reason"],
            "train_seconds":ti["train_seconds"],
            "peak_vram_bytes":ti["peak_vram_bytes"],
            "latency_ms_median":lat,
            **ev,
            "full_reference_rmse":float(full["rmse"]),
            "full_reference_latency_ms":float(full["latency_ms_median"]),
            "core_reference_rmse":float(core["rmse"]),
        }
        row["delta_rmse_pct_vs_full"]=100*(row["rmse"]-row["full_reference_rmse"])/row["full_reference_rmse"]
        row["latency_change_pct_vs_full"]=100*(row["latency_ms_median"]-row["full_reference_latency_ms"])/row["full_reference_latency_ms"]
        save_json(row,rp)
    except Exception as e:
        row={
            "status":"FAILED","tier":tier,"variant":variant,
            "task":task,"seed":seed,"error":repr(e)
        }
        save_json(row,rp)

    sync_finished(run_id,rp,hp,cp)
    if row.get("status")=="COMPLETED":
        cp.unlink(missing_ok=True)
    del m
    torch.cuda.empty_cache(); gc.collect()

# rebuild
rows=[]
for p in OUT.glob("A3R1__*_result.json"):
    try: rows.append(json.loads(p.read_text(encoding="utf-8")))
    except Exception: pass
run_df=pd.DataFrame(rows)
run_df.to_csv(OUT/"A3_R1_run_results.csv",index=False)

main_df=run_df[run_df.tier=="MAIN"].copy()
supp_df=run_df[run_df.tier=="SUPP"].copy()
print("MAIN",len(main_df),"/",len(MAIN_VARIANTS)*len(TASKS)*len(SEEDS))
print("SUPP",len(supp_df),"/",len(SUPP_VARIANTS)*len(TASKS)*len(SUPP_SEEDS))
if (
    len(main_df)!=len(MAIN_VARIANTS)*len(TASKS)*len(SEEDS)
    or not (main_df.status=="COMPLETED").all()
    or len(supp_df)!=len(SUPP_VARIANTS)*len(TASKS)*len(SUPP_SEEDS)
    or not (supp_df.status=="COMPLETED").all()
):
    raise RuntimeError("A3-R1 incomplete; restart notebook and HF resume.")


## 8. Conditional attribution analysis

In [ ]:

# ============================================================
# 8. Analysis
# ============================================================
# Main component effects. MinimalComplete is cumulative, not a single component.
attrib=main_df.copy()
attrib["removal_hurts"]=attrib["delta_rmse_pct_vs_full"]>0

task_comp=attrib.groupby(["task","family","variant"]).agg(
    delta_rmse_pct_mean=("delta_rmse_pct_vs_full","mean"),
    delta_rmse_pct_std=("delta_rmse_pct_vs_full","std"),
    removal_hurts_seed_count=("removal_hurts","sum"),
    rmse_mean=("rmse","mean"),
    full_rmse_mean=("full_reference_rmse","mean"),
    latency_change_pct_mean=("latency_change_pct_vs_full","mean"),
    logspec_high_mean=("logspec_rmse_high","mean"),
).reset_index()

# Full reference spectral metrics are in Base A2 result JSONs.
full_spec=pd.DataFrame([
    {
        "task":t,"seed":s,
        "full_logspec_high":float(FULL_REF[(t,s)]["logspec_rmse_high"])
    }
    for t in TASKS for s in SEEDS
])
tmp=attrib.merge(full_spec,on=["task","seed"])
tmp["delta_logspec_high_pct_vs_full"]=100*(
    tmp.logspec_rmse_high-tmp.full_logspec_high
)/tmp.full_logspec_high

spec_effect=tmp.groupby(["task","variant"]).agg(
    delta_logspec_high_pct_mean=("delta_logspec_high_pct_vs_full","mean"),
    delta_logspec_high_pct_std=("delta_logspec_high_pct_vs_full","std"),
).reset_index()

single_components=[
    "NoPhase","NoSkew","NoFlux","NoHidden","NoSOC","NoBER"
]
global_comp=task_comp[task_comp.variant.isin(single_components)].groupby("variant").agg(
    mean_delta_rmse_pct=("delta_rmse_pct_mean","mean"),
    median_delta_rmse_pct=("delta_rmse_pct_mean","median"),
    positive_task_count=("delta_rmse_pct_mean",lambda x:int((x>0).sum())),
    negative_task_count=("delta_rmse_pct_mean",lambda x:int((x<0).sum())),
    mean_latency_change_pct=("latency_change_pct_mean","mean"),
).reset_index()

# seed-pair consistency over all 5 tasks × 3 seeds = 15
pair_cons=attrib[attrib.variant.isin(single_components)].groupby("variant").agg(
    positive_seed_task_pairs=("removal_hurts","sum"),
    total_seed_task_pairs=("removal_hurts","count"),
).reset_index()
global_comp=global_comp.merge(pair_cons,on="variant")

def classify(r):
    if r.positive_task_count>=4 and r.positive_seed_task_pairs>=12:
        return "broadly_beneficial"
    if r.negative_task_count>=4:
        return "mostly_redundant_or_adverse"
    return "regime_specific"
global_comp["conditional_role"]=global_comp.apply(classify,axis=1)

# Minimal vs Full vs Core
minimal=attrib[attrib.variant=="MinimalComplete"].copy()
minimal_cmp=minimal[[
    "task","seed","rmse","full_reference_rmse","core_reference_rmse",
    "delta_rmse_pct_vs_full"
]].rename(columns={"rmse":"minimal_rmse"})

task_comp.to_csv(OUT/"A3_R1_attribution_task_component.csv",index=False)
global_comp.to_csv(OUT/"A3_R1_component_global_summary.csv",index=False)
spec_effect.to_csv(OUT/"A3_R1_spectral_component_effect.csv",index=False)
minimal_cmp.to_csv(OUT/"A3_R1_minimal_vs_full_vs_core.csv",index=False)
supp_df.to_csv(OUT/"A3_R1_supplement_spectral_readout.csv",index=False)

display(task_comp.sort_values(["task","delta_rmse_pct_mean"],ascending=[True,False]))
display(global_comp.sort_values("mean_delta_rmse_pct",ascending=False))

REPORT["task_component"]=task_comp.to_dict("records")
REPORT["global_component"]=global_comp.to_dict("records")


## 9. Publication figures

In [ ]:

# ============================================================
# 9. Publication-grade figures
# ============================================================
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

plt.rcParams.update({
    "figure.facecolor":"white","axes.facecolor":"white","savefig.facecolor":"white",
    "font.family":"serif",
    "font.serif":["Times New Roman","Nimbus Roman","DejaVu Serif"],
    "mathtext.fontset":"stix","font.size":9,
    "axes.linewidth":0.8,"xtick.direction":"in","ytick.direction":"in",
    "legend.frameon":False,"pdf.fonttype":42,"ps.fonttype":42,
})

COMP_ORDER=["NoPhase","NoSkew","NoFlux","NoHidden","NoSOC","NoBER"]
COMP_LABEL={
    "NoPhase":"Phase encoder",
    "NoSkew":"Skew rotation",
    "NoFlux":"Antisymmetric flux",
    "NoHidden":"Hidden reservoir",
    "NoSOC":"SOC spectral core",
    "NoBER":"BER",
}
TASK_ORDER=list(TASKS)
TASK_LABEL={
    "advection_1d":"Advection",
    "burgers_1d":"Burgers",
    "reacdiff_2d":"2D Reac.-Diff.",
    "swe_2d":"Shallow Water",
    "cfd_2d":"2D CFD",
}

def saveall(fig,stem):
    for ext in ["pdf","svg","png"]:
        kw={"bbox_inches":"tight"}
        if ext=="png": kw["dpi"]=600
        fig.savefig(FIG/f"{stem}.{ext}",**kw)
    plt.close(fig)

# 1. component × regime heatmap
mat=task_comp[
    task_comp.variant.isin(COMP_ORDER)
].pivot(index="variant",columns="task",values="delta_rmse_pct_mean").loc[
    COMP_ORDER,TASK_ORDER
]
lim=max(10,float(np.nanpercentile(np.abs(mat.values),95)))
fig,ax=plt.subplots(figsize=(6.8,3.2))
im=ax.imshow(
    mat.values,aspect="auto",cmap="RdBu_r",
    norm=TwoSlopeNorm(vmin=-lim,vcenter=0,vmax=lim)
)
ax.set_xticks(range(len(TASK_ORDER)))
ax.set_xticklabels([TASK_LABEL[t] for t in TASK_ORDER],rotation=30,ha="right")
ax.set_yticks(range(len(COMP_ORDER)))
ax.set_yticklabels([COMP_LABEL[v] for v in COMP_ORDER])
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        v=mat.iloc[i,j]
        ax.text(j,i,f"{v:+.1f}%",ha="center",va="center",fontsize=7)
cb=fig.colorbar(im,ax=ax,pad=.02)
cb.set_label(r"$\Delta$RMSE after removal (%)")
ax.set_title("Conditional component contribution across dynamics regimes")
fig.tight_layout()
mat.reset_index().to_csv(FIGDATA/"Fig_A3_01_component_regime_heatmap.csv",index=False)
saveall(fig,"Fig_A3_01_component_regime_heatmap")

# 2. global effects
g=global_comp.set_index("variant").loc[COMP_ORDER].reset_index()
fig,ax=plt.subplots(figsize=(5.4,3.0))
x=np.arange(len(g))
bars=ax.bar(
    x,g.mean_delta_rmse_pct,
    edgecolor="black",linewidth=.45
)
ax.axhline(0,color="black",lw=.7)
ax.set_xticks(x)
ax.set_xticklabels([COMP_LABEL[v] for v in g.variant],rotation=35,ha="right")
ax.set_ylabel("Mean conditional ΔRMSE (%)")
for i,r in g.iterrows():
    ax.text(
        i,r.mean_delta_rmse_pct,
        f"{int(r.positive_seed_task_pairs)}/15",
        ha="center",
        va="bottom" if r.mean_delta_rmse_pct>=0 else "top",
        fontsize=7
    )
fig.tight_layout()
g.to_csv(FIGDATA/"Fig_A3_02_component_global_effect.csv",index=False)
saveall(fig,"Fig_A3_02_component_global_effect")

# 3. effect vs latency cost
fig,ax=plt.subplots(figsize=(4.2,3.1))
for _,r in g.iterrows():
    ax.scatter(
        -r.mean_latency_change_pct,
        r.mean_delta_rmse_pct,
        s=42,edgecolor="black",linewidth=.4
    )
    ax.annotate(
        COMP_LABEL[r.variant],
        (-r.mean_latency_change_pct,r.mean_delta_rmse_pct),
        xytext=(4,3),textcoords="offset points",fontsize=7
    )
ax.axhline(0,color="black",lw=.6)
ax.axvline(0,color="black",lw=.6)
ax.set_xlabel("Latency saved when removed (%)")
ax.set_ylabel("RMSE penalty when removed (%)")
fig.tight_layout()
saveall(fig,"Fig_A3_03_effect_vs_latency_cost")

# 4. Full vs MinimalComplete vs Core
rows=[]
for task in TASK_ORDER:
    full=np.mean([FULL_REF[(task,s)]["rmse"] for s in SEEDS])
    core=np.mean([CORE_REF[(task,s)]["rmse"] for s in SEEDS])
    mini=minimal[minimal.task==task].rmse.mean()
    rows.append({"task":task,"Full":full,"MinimalComplete":mini,"Core":core})
cmp=pd.DataFrame(rows)

fig,ax=plt.subplots(figsize=(6.2,3.0))
x=np.arange(len(cmp)); w=.24
ax.bar(x-w,cmp.Full,w,label="CIDM-Complete")
ax.bar(x,cmp.MinimalComplete,w,label="MinimalComplete")
ax.bar(x+w,cmp.Core,w,label="CIDM-Core")
ax.set_yscale("log")
ax.set_xticks(x)
ax.set_xticklabels([TASK_LABEL[t] for t in cmp.task],rotation=30,ha="right")
ax.set_ylabel("RMSE")
ax.legend(ncol=3,fontsize=7)
fig.tight_layout()
cmp.to_csv(FIGDATA/"Fig_A3_04_full_minimal_core.csv",index=False)
saveall(fig,"Fig_A3_04_full_minimal_core")

# 5. spectral effect heatmap
sm=spec_effect[
    spec_effect.variant.isin(COMP_ORDER)
].pivot(index="variant",columns="task",values="delta_logspec_high_pct_mean").loc[
    COMP_ORDER,TASK_ORDER
]
slim=max(10,float(np.nanpercentile(np.abs(sm.values),95)))
fig,ax=plt.subplots(figsize=(6.8,3.2))
im=ax.imshow(
    sm.values,aspect="auto",cmap="RdBu_r",
    norm=TwoSlopeNorm(vmin=-slim,vcenter=0,vmax=slim)
)
ax.set_xticks(range(len(TASK_ORDER)))
ax.set_xticklabels([TASK_LABEL[t] for t in TASK_ORDER],rotation=30,ha="right")
ax.set_yticks(range(len(COMP_ORDER)))
ax.set_yticklabels([COMP_LABEL[v] for v in COMP_ORDER])
cb=fig.colorbar(im,ax=ax,pad=.02)
cb.set_label("High-band log-spectrum error change (%)")
fig.tight_layout()
sm.reset_index().to_csv(FIGDATA/"Fig_A3_05_spectral_effect_heatmap.csv",index=False)
saveall(fig,"Fig_A3_05_spectral_effect_heatmap")

# 6. supplementary spectral readout (seed17 only)
sr=supp_df.copy()
fig,ax=plt.subplots(figsize=(5.8,2.9))
x=np.arange(len(TASK_ORDER))
vals=[]
for task in TASK_ORDER:
    r=sr[sr.task==task].iloc[0]
    vals.append(r.delta_rmse_pct_vs_full)
ax.bar(x,vals,edgecolor="black",linewidth=.45)
ax.axhline(0,color="black",lw=.7)
ax.set_xticks(x)
ax.set_xticklabels([TASK_LABEL[t] for t in TASK_ORDER],rotation=30,ha="right")
ax.set_ylabel("ΔRMSE without spectral readout (%)")
ax.set_title("Supplementary implementation audit (seed 17)")
fig.tight_layout()
saveall(fig,"Fig_A3_06_supp_spectral_readout")


## 10. Final gate + package

In [ ]:

# ============================================================
# 10. Gate + package
# ============================================================
checks={
    "archive_hash_exact":sha256_file(FROZEN/"cidm_architectures.py")==EXPECTED_ARCHIVE_SHA256,
    "five_exact_A2_caches":len(CACHES)==5,
    "thirty_A2_references":len(ref_df)==30,
    "nominal_parameter_equality":bool(param_df["equal"].all()),
    "qualification_all_pass":bool(qual_df["pass"].all()),
    "main_105of105":len(main_df)==105 and bool((main_df.status=="COMPLETED").all()),
    "supp_5of5":len(supp_df)==5 and bool((supp_df.status=="COMPLETED").all()),
    "attribution_written":(OUT/"A3_R1_attribution_task_component.csv").exists(),
    "global_summary_written":(OUT/"A3_R1_component_global_summary.csv").exists(),
    "figures_complete":all(
        (FIG/f"{stem}.{ext}").exists()
        for stem in [
            "Fig_A3_01_component_regime_heatmap",
            "Fig_A3_02_component_global_effect",
            "Fig_A3_03_effect_vs_latency_cost",
            "Fig_A3_04_full_minimal_core",
            "Fig_A3_05_spectral_effect_heatmap",
            "Fig_A3_06_supp_spectral_readout",
        ]
        for ext in ["pdf","svg","png"]
    ),
}

verdict=(
    "PASS_A3_R1_CONDITIONAL_COMPONENT_ATTRIBUTION"
    if all(checks.values())
    else "A3_R1_INCOMPLETE"
)
REPORT["gate"]=checks
REPORT["verdict"]=verdict
REPORT["finished_at"]=time.strftime("%Y-%m-%d %H:%M:%S")
save_json(REPORT,OUT/"CIDM_Foundation_A3_R1_REPORT.json")

gate=OUT/"A3_R1_gate_report.txt"
gate.write_text(
    "\n".join(
        ["CIDM Foundation A3-R1 — Conditional Component Attribution",
         "="*86,f"VERDICT: {verdict}",""]
        +[f"[{'PASS' if v else 'FAIL'}] {k}" for k,v in checks.items()]
        +["",
          "A3-R1 estimates conditional leave-one-out contribution.",
          "Component effects are not additive.",
          "Next: A3-R2 generic capacity rescue only for components with strong attributable effects."]
    ),
    encoding="utf-8"
)
print(gate.read_text(encoding="utf-8"))

zip_path=OUT/"CIDM_Foundation_A3_R1_ComponentAttribution_RESULTS.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as zf:
    for p in OUT.rglob("*"):
        if not p.is_file() or p==zip_path:
            continue
        if "checkpoints" in p.parts:
            continue
        zf.write(p,p.relative_to(OUT))
sha=sha256_file(zip_path)
(OUT/"CIDM_Foundation_A3_R1_ComponentAttribution_RESULTS.zip.sha256").write_text(
    sha+"\n",encoding="utf-8"
)
print("ZIP",zip_path)
print("SHA256",sha)

# Final HF snapshot
try:
    manifest(None,None,"FINAL_PACKAGE")
    retry(
        HF_API.upload_folder,
        repo_id=HF_DATASET_REPO_ID,
        repo_type="dataset",
        folder_path=str(OUT),
        path_in_repo=HF_PREFIX,
        ignore_patterns=["checkpoints/*"],
        commit_message="[A3-R1] final conditional component attribution package",
    )
    print(
        f"https://huggingface.co/datasets/{HF_DATASET_REPO_ID}/tree/main/{HF_PREFIX}"
    )
except Exception as e:
    print("[HF final warning]",repr(e))

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print("Auto-download unavailable outside Colab:",repr(e))
